In [2]:
from sqdtoolz.ExperimentConfiguration import*
from sqdtoolz.Laboratory import*
from sqdtoolz.Experiment import*
from sqdtoolz.Utilities.FileIO import*

from sqdtoolz.Drivers.dummyGENmwSource import*
from sqdtoolz.HAL.ACQ import*
from sqdtoolz.HAL.AWG import*
from sqdtoolz.HAL.DDG import*
from sqdtoolz.HAL.GENmwSource import*

from sqdtoolz.HAL.WaveformGeneric import*
from sqdtoolz.HAL.WaveformMapper import*


from sqdtoolz.HAL.Processors.ProcessorCPU import*
from sqdtoolz.HAL.Processors.CPU.CPU_DDC import*
from sqdtoolz.HAL.Processors.CPU.CPU_FIR import*
from sqdtoolz.HAL.Processors.CPU.CPU_Mean import*

from sqdtoolz.ExperimentSweeps import*

import numpy as np
import shutil
import os.path
import time

import unittest

lab = Laboratory('../UnitTests/UTestExperimentConfiguration.yaml', 'test_save_dir/', using_VS_Code=True)

lab.load_instrument('virACQ')
lab.load_instrument('virDDG')
lab.load_instrument('virAWG')
lab.load_instrument('virMWS')
lab.load_instrument('virMWS2')
lab.load_instrument('virSMU')

#Initialise test-modules
hal_acq = ACQ("dum_acq", lab, 'virACQ')
hal_ddg = DDG("ddg", lab, 'virDDG', )
awg_wfm = WaveformAWG("Wfm1", lab, [('virAWG', 'CH1'), ('virAWG', 'CH2')], 1e9)
hal_mw = GENmwSource("MW-Src", lab, 'virMWS', 'CH1')
hal_mw2 = GENmwSource("MW-Src2", lab, 'virMWS2', 'CH1')
hal_smu = GENsmu('SMU', lab, 'virSMU')

#Reinitialise the waveform
read_segs = []
read_segs2 = []
awg_wfm.clear_segments()
awg_wfm.add_waveform_segment(WFS_Constant("SEQPAD", None, 10e-9, 0.0))
for m in range(4):
    awg_wfm.add_waveform_segment(WFS_Gaussian(f"init{m}", None, 20e-9, 0.5-0.1*m))
    awg_wfm.add_waveform_segment(WFS_Constant(f"zero1{m}", None, 30e-9, 0.1*m))
    awg_wfm.add_waveform_segment(WFS_Gaussian(f"init2{m}", None, 45e-9, 0.5-0.1*m))
    awg_wfm.add_waveform_segment(WFS_Constant(f"zero2{m}", None, 77e-9*(m+1), 0.0))
    read_segs += [f"init{m}"]
    read_segs2 += [f"zero2{m}"]
awg_wfm.get_output_channel(0).marker(1).set_markers_to_segments(read_segs)
awg_wfm.get_output_channel(1).marker(0).set_markers_to_segments(read_segs2)
awg_wfm.AutoCompression = 'None'#'Basic'

expConfig = ExperimentConfiguration('testConf', lab, 1.0, ['ddg', 'Wfm1', 'MW-Src'], 'dum_acq')

VariableInternal('myFreq', lab)
VariableProperty('testAmpl', lab, lab.HAL("Wfm1").get_waveform_segment('init0'), 'Amplitude')
lab.VAR('testAmpl').Value = 86
VariableProperty('test RepTime', lab, lab.HAL("ddg"), 'RepetitionTime')
lab.VAR('test RepTime').Value = 99
VariableInternal('myDura1', lab)
lab.VAR('myDura1').Value = 2016
VariableProperty('myDura2', lab, lab.HAL("Wfm1").get_waveform_segment('init2'), 'Duration')
VariableSpaced('testSpace', lab, 'myDura1', 'myDura2', 3.1415926)
lab.VAR('testSpace').Value = 2016

WFMT_ModulationIQ("IQmod", lab, 49e6)
lab.WFMT("IQmod").IQFrequency = 84e7
lab.WFMT("IQmod").IQAmplitude = 9.4
lab.WFMT("IQmod").IQAmplitudeFactor = 78.1
lab.WFMT("IQmod").IQPhaseOffset = 54.3
lab.WFMT("IQmod").IQdcOffset = (9,1)
lab.WFMT("IQmod").IQUpperSideband = False

In [ ]:
exp = Experiment("test", lab.CONFIG('testConf'))
lab.VAR('myFreq').Value = 5
res = lab.run_single(exp, [(lab.VAR("testAmpl"), np.arange(0,3,0.01))], delay=1, rec_params=[lab.VAR('myFreq'), lab.VAR('testAmpl')])
